In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier


# =========================================================
# 1. 데이터 불러오기
# =========================================================

train = pd.read_csv(
    "/kaggle/input/competitions/titanic/train.csv"
)

test = pd.read_csv(
    "/kaggle/input/competitions/titanic/test.csv"
)

y = train["Survived"]


# =========================================================
# 2. Random Forest 기본 변수
# =========================================================


features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

numeric_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]


# =========================================================
# 3. 결측값 처리 및 One-Hot Encoding
# =========================================================

numeric_transformer = SimpleImputer(
    strategy="median",
    add_indicator=True
)

categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_transformer,
        numeric_features
    ),
    (
        "categorical",
        categorical_transformer,
        categorical_features
    )
])


# =========================================================
# 4. Random Forest 모델
# =========================================================

model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "random_forest",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        )
    )
])


# =========================================================
# 5. 전체 train 데이터로 학습
# =========================================================

model.fit(
    train[features],
    y
)


# =========================================================
# 6. test 승객의 기본 생존 확률
# =========================================================

#각 승객이 0또는 1일 확률을 계산
base_probability = model.predict_proba(
    test[features]
)[:, 1]#생존 확률만 취득 
#-> base_probability에는 각 승객의 기본 생존 확률 저장

# =========================================================
# 7. 가족 그룹 키 생성 함수
# =========================================================

def make_family_key(data):
    result = data.copy()

    # 이름 앞부분에서 성씨 추출
    surname = (
        result["Name"]
        .str.split(",")
        .str[0]
        .str.strip()
    )

    # 본인을 포함한 가족 규모
    family_size = (
        result["SibSp"]
        + result["Parch"]
        + 1
    )

    # 성씨와 가족 규모를 결합
    family_key = (
        surname
        + "_"
        + family_size.astype(str)
    )

    # 혼자인 승객은 가족 정보 사용 안 함
    family_key = family_key.where(
        family_size > 1,
        np.nan
    )

    return family_key


train_family_key = make_family_key(train)
test_family_key = make_family_key(test)


# =========================================================
# 8. 여성·어린이 가족 구성원 선택
# =========================================================

train_protected_group = (
    (train["Sex"] == "female")
    | (train["Age"] < 16)
)

test_protected_group = (
    (test["Sex"] == "female")
    | (test["Age"] < 16)
)


# =========================================================
# 9. train 가족별 생존 통계 계산
# =========================================================

family_data = pd.DataFrame({
    "FamilyKey": train_family_key,
    "Survived": y,
    "ProtectedGroup": train_protected_group
})

# 여성과 어린이의 가족 정보만 사용
family_data = family_data[
    family_data["ProtectedGroup"]
].dropna(#FamilyKey가 없는 승객 제거
    subset=["FamilyKey"]
)

family_statistics = (
    family_data
    .groupby("FamilyKey")["Survived"]
    .agg(["sum", "count"])
)


# =========================================================
# 10. 가족 생존율 smoothing
# =========================================================
"""
가족 데이터가 1명뿐이면 결과가 너무 극단적일 수 있음
"""
global_survival_rate = y.mean()

# 전체 생존율을 가상 표본 2명만큼 추가
family_survival_rate = (
    family_statistics["sum"]
    + 2 * global_survival_rate
) / (
    family_statistics["count"]
    + 2
)


# =========================================================
# 11. test 승객에 가족 생존율 연결
# =========================================================

test_family_probability = (
    test_family_key
    .map(family_survival_rate)
)

# 가족 정보 적용 조건:
# 1. 여성 또는 16세 미만
# 2. train에서 같은 가족 정보가 발견됨
use_family_information = (
    test_protected_group
    & test_family_probability.notna()
)


# =========================================================
# 12. 기본 확률과 가족 확률 결합
# =========================================================

#가족 정보가 없는 승객은 기본 확률사용
final_probability = base_probability.copy()

#가족 정보가 있는 승객은 RF가 예측한 확률의 50% + 가족 생존 확률의 50% 사용
final_probability[use_family_information] = (
    0.5
    * base_probability[use_family_information]
    + 0.5
    * test_family_probability[
        use_family_information
    ].to_numpy()
)


# =========================================================
# 13. Threshold 0.5로 최종 예측
# =========================================================

#최종 생존 확률이 0.5 이상이면 생존으로 예측
final_prediction = (
    final_probability >= 0.5
).astype(int)

base_prediction = (
    base_probability >= 0.5
).astype(int)


# =========================================================
# 14. 제출 파일 생성
# =========================================================

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": final_prediction
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)


# =========================================================
# 15. 결과 확인
# =========================================================

print("제출 파일 크기:", submission.shape)

print(
    "가족 정보가 적용된 승객:",
    int(use_family_information.sum())
)

print(
    "기본 RF와 예측이 달라진 승객:",
    int((base_prediction != final_prediction).sum())
)

print("\n최종 예측 분포:")
print(
    submission["Survived"]
    .value_counts()
    .sort_index()
)

print("\n제출 파일 검증:")
print(submission.columns.tolist())
print(submission.isna().sum())

display(submission.head())

제출 파일 크기: (418, 2)
가족 정보가 적용된 승객: 51
기본 RF와 예측이 달라진 승객: 6

최종 예측 분포:
Survived
0    284
1    134
Name: count, dtype: int64

제출 파일 검증:
['PassengerId', 'Survived']
PassengerId    0
Survived       0
dtype: int64


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
